In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import mlflow
import mlflow.xgboost

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report
)

from xgboost import XGBClassifier

In [4]:
df = pd.read_csv("../data/raw/creditcard.csv")

print("Dataset shape:", df.shape)
print("Fraud cases:", df["Class"].sum())
print("Normal cases:", (df["Class"] == 0).sum())

Dataset shape: (284807, 31)
Fraud cases: 492
Normal cases: 284315


In [5]:
X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (227845, 30)
Testing: (56962, 30)


In [6]:
X_train = X_train.copy()
X_test = X_test.copy()

# Transaction hour
X_train["TransactionHour"] = (
    X_train["Time"] // 3600
) % 24

X_test["TransactionHour"] = (
    X_test["Time"] // 3600
) % 24

# Cyclical encoding
X_train["HourSin"] = np.sin(
    2 * np.pi * X_train["TransactionHour"] / 24
)

X_train["HourCos"] = np.cos(
    2 * np.pi * X_train["TransactionHour"] / 24
)

X_test["HourSin"] = np.sin(
    2 * np.pi * X_test["TransactionHour"] / 24
)

X_test["HourCos"] = np.cos(
    2 * np.pi * X_test["TransactionHour"] / 24
)

# Log-transformed amount
X_train["AmountLog"] = np.log1p(
    X_train["Amount"]
)

X_test["AmountLog"] = np.log1p(
    X_test["Amount"]
)

# Amount bucket
bins = [-1, 10, 50, 100, 500, 1000, np.inf]

X_train["AmountBucket"] = pd.cut(
    X_train["Amount"],
    bins=bins,
    labels=False
)

X_test["AmountBucket"] = pd.cut(
    X_test["Amount"],
    bins=bins,
    labels=False
)

In [7]:
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

Training features: (227845, 35)
Testing features: (56962, 35)


In [8]:
print(
    list(X_train.columns) == list(X_test.columns)
)

True


In [9]:
negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

scale_pos_weight = negative / positive

print("Negative:", negative)
print("Positive:", positive)
print("scale_pos_weight:", scale_pos_weight)

Negative: 227451
Positive: 394
scale_pos_weight: 577.2868020304569


In [10]:
mlflow.set_experiment(
    "Fraud Detection - XGBoost"
)

<Experiment: artifact_location='file:d:/vscode/real-time_fraud_detection/notebook/mlruns/1', creation_time=1790490699515, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1790490699515, lifecycle_stage='active', name='Fraud Detection - XGBoost', tags={}, trace_location=None, workspace='default'>

In [11]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

In [12]:
with mlflow.start_run():

    # Train model
    xgb_model.fit(
        X_train,
        y_train
    )

    # Predictions
    y_pred = xgb_model.predict(X_test)

    y_prob = xgb_model.predict_proba(
        X_test
    )[:, 1]

    # Metrics
    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        y_prob
    )

    pr_auc = average_precision_score(
        y_test,
        y_prob
    )

    # Log parameters
    mlflow.log_params({
        "n_estimators": 200,
        "max_depth": 5,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "scale_pos_weight": scale_pos_weight
    })

    # Log metrics
    mlflow.log_metrics({
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc
    })

    # Log model
    mlflow.xgboost.log_model(
        xgb_model,
        name="xgboost_model"
    )

    print("Run completed")

Run completed


In [13]:
mlflow.log_params({
    "training_rows": len(X_train),
    "test_rows": len(X_test),
    "num_features": X_train.shape[1],
    "fraud_training_samples": int(y_train.sum()),
    "feature_engineering": "Time + HourSin + HourCos + AmountLog + AmountBucket"
})

In [14]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred
)

plt.title("XGBoost Confusion Matrix")
plt.tight_layout()

plt.savefig(
    "../models/confusion_matrix.png"
)

plt.close()

print("Confusion matrix saved.")

Confusion matrix saved.


In [15]:
mlflow.log_artifact(
    "../models/confusion_matrix.png"
)

print("Confusion matrix logged to MLflow.")

Confusion matrix logged to MLflow.


In [16]:
importance = pd.Series(
    xgb_model.feature_importances_,
    index=X_train.columns
).sort_values(
    ascending=False
)

plt.figure(figsize=(10, 6))

importance.head(15).sort_values().plot(
    kind="barh"
)

plt.title("Top 15 XGBoost Features")
plt.xlabel("Importance")

plt.tight_layout()

plt.savefig(
    "../models/feature_importance.png"
)

plt.close()

print("Feature importance saved.")

Feature importance saved.


In [17]:
mlflow.log_artifact(
    "../models/feature_importance.png"
)

print("Feature importance logged to MLflow.")

Feature importance logged to MLflow.


In [18]:
feature_list = "\n".join(X_train.columns)

with open(
    "../models/feature_list.txt",
    "w"
) as f:
    f.write(feature_list)

In [19]:
mlflow.log_artifact(
    "../models/feature_list.txt"
)

In [20]:
def train_and_log_xgb(
    run_name,
    n_estimators,
    max_depth,
    learning_rate,
    subsample=0.8,
    colsample_bytree=0.8
):

    model = XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
    )

    with mlflow.start_run(run_name=run_name):

        # Train
        model.fit(
            X_train,
            y_train
        )

        # Predictions
        y_pred = model.predict(X_test)

        y_prob = model.predict_proba(
            X_test
        )[:, 1]

        # Metrics
        precision = precision_score(
            y_test,
            y_pred,
            zero_division=0
        )

        recall = recall_score(
            y_test,
            y_pred,
            zero_division=0
        )

        f1 = f1_score(
            y_test,
            y_pred,
            zero_division=0
        )

        roc_auc = roc_auc_score(
            y_test,
            y_prob
        )

        pr_auc = average_precision_score(
            y_test,
            y_prob
        )

        # Parameters
        mlflow.log_params({
            "model": "XGBoost",
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "learning_rate": learning_rate,
            "subsample": subsample,
            "colsample_bytree": colsample_bytree,
            "scale_pos_weight": scale_pos_weight,
            "training_rows": len(X_train),
            "test_rows": len(X_test),
            "num_features": X_train.shape[1]
        })

        # Metrics
        mlflow.log_metrics({
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "roc_auc": roc_auc,
            "pr_auc": pr_auc
        })

        # Log model
        mlflow.xgboost.log_model(
            model,
            name="xgboost_model"
        )

        print(f"Run: {run_name}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall:    {recall:.4f}")
        print(f"F1:        {f1:.4f}")
        print(f"ROC-AUC:   {roc_auc:.4f}")
        print(f"PR-AUC:    {pr_auc:.4f}")

    return model

In [22]:
mlflow.active_run()

<ActiveRun: >

In [23]:
mlflow.end_run()

In [24]:
print(mlflow.active_run())

None


In [25]:
model_2 = train_and_log_xgb(
    run_name="XGBoost_Deeper",
    n_estimators=200,
    max_depth=7,
    learning_rate=0.05
)

Run: XGBoost_Deeper
Precision: 0.8119
Recall:    0.8367
F1:        0.8241
ROC-AUC:   0.9787
PR-AUC:    0.8667


In [26]:
model_3 = train_and_log_xgb(
    run_name="XGBoost_Lower_Learning_Rate",
    n_estimators=300,
    max_depth=5,
    learning_rate=0.03
)

Run: XGBoost_Lower_Learning_Rate
Precision: 0.5484
Recall:    0.8673
F1:        0.6719
ROC-AUC:   0.9823
PR-AUC:    0.8460


In [27]:
model_4 = train_and_log_xgb(
    run_name="XGBoost_More_Trees",
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05
)

Run: XGBoost_More_Trees
Precision: 0.8557
Recall:    0.8469
F1:        0.8513
ROC-AUC:   0.9835
PR-AUC:    0.8746


In [28]:
runs = mlflow.search_runs(
    experiment_names=["Fraud Detection - XGBoost"]
)

runs[
    [
        "run_id",
        "tags.mlflow.runName",
        "metrics.precision",
        "metrics.recall",
        "metrics.f1",
        "metrics.roc_auc",
        "metrics.pr_auc"
    ]
]

,run_id,tags.mlflow.runName,metrics.precision,metrics.recall,metrics.f1,metrics.roc_auc,metrics.pr_auc
0,d3cbd465de6a49198e054b06368883eb,XGBoost_More_Trees,0.855670,0.846939,0.851282,0.983464,0.874552
1,59aa0c3633474635b9f4652557f155d1,XGBoost_Lower_Learning_Rate,0.548387,0.867347,0.671937,0.982317,0.845964
2,571194bb1f724162805859ec48aba4a0,XGBoost_Deeper,0.811881,0.836735,0.824121,0.978717,0.866723
3,5b689e8152814e9fada1474c07bcf3b6,receptive-mare-681,NaN,NaN,NaN,NaN,NaN
4,e7f2b3055fd7419b9e730217d2c37626,traveling-koi-933,0.611511,0.867347,0.717300,0.981635,0.852867
5,63e1d4207b864b49aed58b354f25ab85,selective-seal-752,NaN,NaN,NaN,NaN,NaN
6,6fb9f6410baa4fe987114c500eb09e6d,legendary-midge-821,0.611511,0.867347,0.717300,0.981635,0.852867


In [29]:
runs_sorted = runs.sort_values(
    "metrics.pr_auc",
    ascending=False
)

runs_sorted[
    [
        "run_id",
        "tags.mlflow.runName",
        "metrics.precision",
        "metrics.recall",
        "metrics.f1",
        "metrics.roc_auc",
        "metrics.pr_auc"
    ]
]

,run_id,tags.mlflow.runName,metrics.precision,metrics.recall,metrics.f1,metrics.roc_auc,metrics.pr_auc
0,d3cbd465de6a49198e054b06368883eb,XGBoost_More_Trees,0.855670,0.846939,0.851282,0.983464,0.874552
2,571194bb1f724162805859ec48aba4a0,XGBoost_Deeper,0.811881,0.836735,0.824121,0.978717,0.866723
4,e7f2b3055fd7419b9e730217d2c37626,traveling-koi-933,0.611511,0.867347,0.717300,0.981635,0.852867
6,6fb9f6410baa4fe987114c500eb09e6d,legendary-midge-821,0.611511,0.867347,0.717300,0.981635,0.852867
1,59aa0c3633474635b9f4652557f155d1,XGBoost_Lower_Learning_Rate,0.548387,0.867347,0.671937,0.982317,0.845964
3,5b689e8152814e9fada1474c07bcf3b6,receptive-mare-681,NaN,NaN,NaN,NaN,NaN
5,63e1d4207b864b49aed58b354f25ab85,selective-seal-752,NaN,NaN,NaN,NaN,NaN


In [30]:
runs = mlflow.search_runs(
    experiment_names=["Fraud Detection - XGBoost"]
)

comparison = runs[
    [
        "run_id",
        "tags.mlflow.runName",
        "metrics.precision",
        "metrics.recall",
        "metrics.f1",
        "metrics.roc_auc",
        "metrics.pr_auc"
    ]
]

comparison

,run_id,tags.mlflow.runName,metrics.precision,metrics.recall,metrics.f1,metrics.roc_auc,metrics.pr_auc
0,d3cbd465de6a49198e054b06368883eb,XGBoost_More_Trees,0.855670,0.846939,0.851282,0.983464,0.874552
1,59aa0c3633474635b9f4652557f155d1,XGBoost_Lower_Learning_Rate,0.548387,0.867347,0.671937,0.982317,0.845964
2,571194bb1f724162805859ec48aba4a0,XGBoost_Deeper,0.811881,0.836735,0.824121,0.978717,0.866723
3,5b689e8152814e9fada1474c07bcf3b6,receptive-mare-681,NaN,NaN,NaN,NaN,NaN
4,e7f2b3055fd7419b9e730217d2c37626,traveling-koi-933,0.611511,0.867347,0.717300,0.981635,0.852867
5,63e1d4207b864b49aed58b354f25ab85,selective-seal-752,NaN,NaN,NaN,NaN,NaN
6,6fb9f6410baa4fe987114c500eb09e6d,legendary-midge-821,0.611511,0.867347,0.717300,0.981635,0.852867


In [31]:
comparison_sorted = comparison.sort_values(
    "metrics.pr_auc",
    ascending=False
)

comparison_sorted

,run_id,tags.mlflow.runName,metrics.precision,metrics.recall,metrics.f1,metrics.roc_auc,metrics.pr_auc
0,d3cbd465de6a49198e054b06368883eb,XGBoost_More_Trees,0.855670,0.846939,0.851282,0.983464,0.874552
2,571194bb1f724162805859ec48aba4a0,XGBoost_Deeper,0.811881,0.836735,0.824121,0.978717,0.866723
4,e7f2b3055fd7419b9e730217d2c37626,traveling-koi-933,0.611511,0.867347,0.717300,0.981635,0.852867
6,6fb9f6410baa4fe987114c500eb09e6d,legendary-midge-821,0.611511,0.867347,0.717300,0.981635,0.852867
1,59aa0c3633474635b9f4652557f155d1,XGBoost_Lower_Learning_Rate,0.548387,0.867347,0.671937,0.982317,0.845964
3,5b689e8152814e9fada1474c07bcf3b6,receptive-mare-681,NaN,NaN,NaN,NaN,NaN
5,63e1d4207b864b49aed58b354f25ab85,selective-seal-752,NaN,NaN,NaN,NaN,NaN


In [32]:
selected_run_id = "d3cbd465de6a49198e054b06368883eb"

print("Selected Run ID:", selected_run_id)

Selected Run ID: d3cbd465de6a49198e054b06368883eb


In [33]:
model_uri = f"runs:/{selected_run_id}/xgboost_model"

loaded_model = mlflow.xgboost.load_model(model_uri)

print("Model loaded successfully!")

Model loaded successfully!


In [34]:
y_pred_loaded = loaded_model.predict(X_test)

y_prob_loaded = loaded_model.predict_proba(X_test)[:, 1]

print("Predictions generated:", len(y_pred_loaded))

Predictions generated: 56962


In [35]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

precision = precision_score(
    y_test,
    y_pred_loaded,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred_loaded,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred_loaded,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_test,
    y_prob_loaded
)

pr_auc = average_precision_score(
    y_test,
    y_prob_loaded
)

print("Precision :", precision)
print("Recall    :", recall)
print("F1 Score  :", f1)
print("ROC-AUC   :", roc_auc)
print("PR-AUC    :", pr_auc)

Precision : 0.8556701030927835
Recall    : 0.8469387755102041
F1 Score  : 0.8512820512820513
ROC-AUC   : 0.9834639469181032
PR-AUC    : 0.8745523959104827


In [36]:
from mlflow import MlflowClient

model_name = "FraudDetectionXGBoost"

result = mlflow.register_model(
    model_uri=model_uri,
    name=model_name
)

print("Model registered successfully!")
print("Model name:", result.name)
print("Model version:", result.version)

Successfully registered model 'FraudDetectionXGBoost'.
2026/09/28 20:11:36 WARNING mlflow.tracking._model_registry.fluent: Run with id d3cbd465de6a49198e054b06368883eb has no artifacts at artifact path 'xgboost_model', registering model based on models:/m-b275afe634ae4494a4f7f7d3d077c168 instead


Model registered successfully!
Model name: FraudDetectionXGBoost
Model version: 1


Created version '1' of model 'FraudDetectionXGBoost'.


In [37]:
loaded_model.save_model("../models/xgboost_model.json")